# Speech Data Augmentation and Classification

This notebook creates a tiny synthetic speech-like task:
- Class 0: low frequency wave
- Class 1: high frequency wave

A/B test:
- baseline
- augmented with noise, shift, and gain

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

In [ ]:
def make_audio_dataset(n_samples=3000, length=1600):
    X, y = [], []
    t = np.linspace(0, 1, length)
    for i in range(n_samples):
        label = i % 2
        freq = 5 if label == 0 else 15
        signal = np.sin(2 * np.pi * freq * t)
        signal += np.random.normal(0, 0.05, size=length)
        X.append(signal.astype(np.float32))
        y.append(label)
    X = np.array(X)[..., None]
    y = np.array(y, dtype=np.int32)
    return X, y

def augment_audio(X):
    X_aug = []
    for x in X:
        sig = x.copy().squeeze()
        sig = sig * np.random.uniform(0.8, 1.2)
        shift = np.random.randint(-30, 31)
        sig = np.roll(sig, shift)
        sig = sig + np.random.normal(0, 0.03, size=sig.shape)
        X_aug.append(sig[:, None].astype(np.float32))
    return np.array(X_aug)

X_train, y_train = make_audio_dataset(2500)
X_test, y_test = make_audio_dataset(800)

X_train_aug = np.concatenate([X_train, augment_audio(X_train[:1200])], axis=0)
y_train_aug = np.concatenate([y_train, y_train[:1200]], axis=0)

In [ ]:
def build_audio_model():
    model = keras.Sequential([
        layers.Input(shape=(1600, 1)),
        layers.Conv1D(16, 9, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 9, activation="relu"),
        layers.MaxPooling1D(2),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

base_model = build_audio_model()
base_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)
base_acc = base_model.evaluate(X_test, y_test, verbose=0)[1]

aug_model = build_audio_model()
aug_model.fit(X_train_aug, y_train_aug, epochs=5, batch_size=64, verbose=0)
aug_acc = aug_model.evaluate(X_test, y_test, verbose=0)[1]

print("Baseline accuracy:", round(base_acc, 4))
print("Augmented accuracy:", round(aug_acc, 4))